#  Hawaiʻi Skilled-Trade Construction Apprenticeship Data Analysis

**Sources:** `hawaii_apprenticeships_clean.parquet`, `national_apprenticeships_clean.parquet`, `hawaii_employment_forecast_clean.parquet`  
**Output:** `hawaii_merged.parquet`, `hawaii_trades.parquet`, `hawaii_program_metrics.parquet`, `hawaii_occ_metrics.parquet`

## 1. Load Libraries

In [23]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
from lifelines import NelsonAalenFitter
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output
from lifelines import AalenJohansenFitter

In [24]:
# ── Toggle these values to change all downstream analysis ─────────────────────
SCOPE = 'trades'   

MIN_N = 20         # Minimum apprentices per program for more reliable program-level

SUPPRESSION_THRESHOLD = 10   # Minimum cell size for privacy control

DATA_AS_OF = pd.Timestamp.today()  # set to data extract date if known

## 2. Load & Merge Data

In [7]:
national_data = pd.read_parquet('../data/processed/national_apprenticeships_clean.parquet')
hawaii_data   = pd.read_parquet('../data/processed/hawaii_apprenticeships_clean.parquet')
hi_forecast   = pd.read_parquet('../data/processed/hawaii_employment_forecast_clean.parquet')

print(f"National records: {len(national_data):,}")
print(f"Hawaii records:   {len(hawaii_data):,}")
print(f"Forecast rows:    {len(hi_forecast):,}")

National records: 1,456,283
Hawaii records:   14,238
Forecast rows:    551


In [8]:
# Detailed occupations only (SOC Level 4) for wage merge
hi_forecast_detailed = hi_forecast[hi_forecast['SOC_Level'] == 4].copy()

hawaii_merged = hawaii_data.merge(
    hi_forecast_detailed,
    left_on  = 'SOC_MERGE_KEY',
    right_on = 'SOC_Code',
    how      = 'left'
)

unmatched_mask = hawaii_merged['Median_Wage'].isna()
unmatched      = hawaii_merged[unmatched_mask].copy()

print(f"Merged rows:  {len(hawaii_merged):,}")
print(f"Wage matched: {(~unmatched_mask).sum():,}")
print(f"Wage missing: {unmatched_mask.sum():,}")

Merged rows:  14,238
Wage matched: 13,293
Wage missing: 945


In [9]:
# SOC crosswalk — manual mapping for codes missing from Hawaii forecast
soc_crosswalk = {
    '47-2171': {'mapped_to_soc': '47-2221', 'mapped_to_title': 'Structural Iron and Steel Workers',
                'reason': 'Direct SOC hierarchy neighbor — both involve structural ironwork', 'confidence': 'high'},
    '47-4021': {'mapped_to_soc': '49-2094', 'mapped_to_title': 'Electrical & Electronics Repairers, Commercial',
                'reason': 'Modern elevator work is predominantly electrical/electronic systems', 'confidence': 'moderate'},
    '51-4192': {'mapped_to_soc': '51-4121', 'mapped_to_title': 'Welders, Cutters, Solderers, and Brazers',
                'reason': 'Shared blueprint reading, metal positioning and fabrication prep tasks', 'confidence': 'moderate'},
    '47-2131': {'mapped_to_soc': '47-2132', 'mapped_to_title': 'Insulation Workers, Mechanical',
                'reason': 'Sister code — near identical materials and techniques', 'confidence': 'high'},
    '47-4099': {'mapped_to_soc': '47-2061', 'mapped_to_title': 'Construction Laborers',
                'reason': 'Catch-all construction category best approximated by general laborers', 'confidence': 'low'},
    '47-2071': {'mapped_to_soc': '47-2073', 'mapped_to_title': 'Operating Engineers and Other Construction Equipment Operators',
                'reason': 'Same class of heavy machinery operation and certifications', 'confidence': 'high'},
    '49-9044': {'mapped_to_soc': '49-9041', 'mapped_to_title': 'Industrial Machinery Mechanics',
                'reason': 'Most similar skill overlap in mechanical, rigging, and precision alignment', 'confidence': 'high'},
    '51-2041': {'mapped_to_soc': '47-2221', 'mapped_to_title': 'Structural Iron and Steel Workers',
                'reason': 'Fabricators and ironworkers share blueprint reading, steel alignment and fastening', 'confidence': 'moderate'},
    '47-2022': {'mapped_to_soc': '47-2021', 'mapped_to_title': 'Brickmasons and Blockmasons',
                'reason': 'Same masonry tools and techniques — only material differs', 'confidence': 'high'},
    '31-1121': {'mapped_to_soc': '31-1120', 'mapped_to_title': 'Home Health and Personal Care Aides',
                'reason': 'Similar non-medical living assistance for elderly or disabled clients', 'confidence': 'high'},
}

wage_lookup = hi_forecast[['SOC_Code', 'Median_Wage']].drop_duplicates('SOC_Code')

unmatched['SOC_CROSSWALK']    = unmatched['SOC_MERGE_KEY'].map({k: v['mapped_to_soc']   for k, v in soc_crosswalk.items()})
unmatched['PROXY_TITLE']      = unmatched['SOC_MERGE_KEY'].map({k: v['mapped_to_title'] for k, v in soc_crosswalk.items()})
unmatched['PROXY_REASON']     = unmatched['SOC_MERGE_KEY'].map({k: v['reason']          for k, v in soc_crosswalk.items()})
unmatched['PROXY_CONFIDENCE'] = unmatched['SOC_MERGE_KEY'].map({k: v['confidence']      for k, v in soc_crosswalk.items()})

unmatched['Median_Wage'] = (
    unmatched.merge(wage_lookup, left_on='SOC_CROSSWALK', right_on='SOC_Code',
                    how='left', suffixes=('', '_new'))['Median_Wage_new'].values
)

hawaii_merged.loc[unmatched_mask, 'Median_Wage']      = unmatched['Median_Wage'].values
hawaii_merged.loc[unmatched_mask, 'PROXY_TITLE']      = unmatched['PROXY_TITLE'].values
hawaii_merged.loc[unmatched_mask, 'PROXY_REASON']     = unmatched['PROXY_REASON'].values
hawaii_merged.loc[unmatched_mask, 'PROXY_CONFIDENCE'] = unmatched['PROXY_CONFIDENCE'].values
hawaii_merged['IS_PROXY'] = hawaii_merged['PROXY_TITLE'].notna()

print(f"Remaining unmatched: {hawaii_merged['Median_Wage'].isna().sum():,}")
print(f"Rows using proxy:    {hawaii_merged['IS_PROXY'].sum():,}")

Remaining unmatched: 44
Rows using proxy:    901


## 3. Helper Function

`completion_rate` uses `n = completed + cancelled` — excludes active apprentices who haven't reached an outcome yet.

Unknown problem:
- `early_exit_rate`   = lower bound (known early exits)
- `cancellation_rate` = upper bound (all cancellations, some timing unknown)

In [10]:
def compute_apprenticeship_metrics(df):
    total        = len(df)
    completed    = (df['APPRENTICE_STATUS_CODE'] == 'Completed').sum()
    cancelled    = (df['APPRENTICE_STATUS_CODE'] == 'Cancelled').sum()
    active       = (df['APPRENTICE_STATUS_CODE'] == 'Active').sum()
    early_exit   = (df['EARLY_TRAINING_EXIT_TYPE'] == 1).sum()
    late_exit    = (df['EARLY_TRAINING_EXIT_TYPE'] == 2).sum()
    unknown_exit = (df['EARLY_TRAINING_EXIT_TYPE'] == -1).sum()
    n = completed + cancelled

    return pd.Series({
        'total_apprentices'               : int(total),
        'active'                          : int(active),
        'completed'                       : int(completed),
        'cancelled'                       : int(cancelled),
        'unknown_exit'                    : int(unknown_exit),
        'completion_rate'                 : round((completed / n) * 100, 2) if n else None,
        'cancellation_rate'               : round((cancelled / n) * 100, 2) if n else None,
        'early_exit_rate'                 : round((early_exit / n) * 100, 2) if n else None,
        'late_exit_rate'                  : round((late_exit / n) * 100, 2) if n else None,
        'completion_to_cancellation_ratio': round((completed / cancelled), 2) if cancelled else None,
    })

## 4. Trade Filtering & Dataset Toggle

**Every chart below uses these variables — changing `SCOPE` and re-running updates everything.**

In [11]:
target_soc = [
    '47-2031',  # Carpenters
    '47-2111',  # Electricians
    '47-2152',  # Plumbers, Pipefitters, and Steamfitters
    '47-2211',  # Sheet Metal Workers
    '47-2021',  # Brickmasons and Blockmasons
    '47-2121',  # Glaziers
    '47-4021',  # Elevator and Escalator Installers
    '47-2221',  # Structural Iron and Steel Workers
    '47-2181',  # Roofers
    '47-2141',  # Painters
    '47-2061',  # Construction Laborers
    '47-2081',  # Drywall Installers
    '47-2044',  # Tile and Stone Setters
    '49-9021',  # HVAC Mechanics
    '47-2171',  # Reinforcing Iron Workers
    '47-2051',  # Cement Masons
    '47-2082',  # Tapers
    '47-2042',  # Floor Layers
    '49-9051',  # Electrical Power-Line Installers
    '47-2161',  # Plasterers
    '47-2073',  # Operating Engineers
    '47-2131',  # Insulation Workers
    '47-2071',  # Paving Operators
    '47-2022',  # Stonemasons
    '47-2011',  # Boilermakers
    '49-9044',  # Millwrights
    '47-2132',  # Insulation Workers, Mechanical
    '47-2231',  # Solar PV Installers
    '49-9052',  # Telecom Line Installers
]

hi_trades       = hawaii_merged[hawaii_merged['ONET_SOC_CD'].fillna('').str[:7].isin(target_soc)].copy()
national_trades = national_data[national_data['ONET_SOC_CD'].fillna('').str[:7].isin(target_soc)].copy()

print(f"Hawaii trade records:   {len(hi_trades):,}")
print(f"National trade records: {len(national_trades):,}")

Hawaii trade records:   11,754
National trade records: 799,681


In [12]:
# Resolve active dataset from SCOPE
if SCOPE == 'trades':
    active_data = hi_trades.copy()
    active_nat  = national_trades.copy()
    scope_label = 'Hawaiʻi Construction & Installation Trades'
else:
    active_data = hawaii_merged.copy()
    active_nat  = national_data.copy()
    scope_label = 'Hawaiʻi All Apprenticeships'

print(f"Active dataset: {scope_label}")
print(f"Records:        {len(active_data):,}")
print(f"Occ. groups:    {active_data['OCCUPATION_GROUP_NAME'].nunique()}")

Active dataset: Hawaiʻi Construction & Installation Trades
Records:        11,754
Occ. groups:    27


## 5. Program-Level Metrics

Grouped by `PROGRAM_ID` — not `PROGRAM_NAME`, which is human-typed and inconsistent.  
Program name is merged back after aggregation for display only.


In [13]:
# Hawaii program-level metrics (active scope)
hi_metrics_by_program = (
    active_data.groupby('PROGRAM_ID')
    .apply(compute_apprenticeship_metrics)
    .reset_index()
    .merge(
        active_data[['PROGRAM_ID', 'PROGRAM_NAME', 'OCCUPATION_GROUP_NAME']]
        .drop_duplicates('PROGRAM_ID'),
        on='PROGRAM_ID', how='left'
    )
)

national_metrics_by_program = (
    active_nat
    .groupby('PROGRAM_ID')
    .apply(compute_apprenticeship_metrics)
    .reset_index()
    .merge(
        active_nat[['PROGRAM_ID', 'OCCUPATION_GROUP_NAME']]
        .drop_duplicates('PROGRAM_ID'),
        on='PROGRAM_ID', how='left'
    )
)

# Then recompute filtered_national
filtered_national = national_metrics_by_program[
    national_metrics_by_program['total_apprentices'] >= MIN_N
].copy()

# National benchmark per occupation group (for deviation charts)
national_benchmark = (
    active_nat
    .groupby('OCCUPATION_GROUP_NAME')
    .apply(compute_apprenticeship_metrics)
    [['completion_rate']]
    .rename(columns={'completion_rate': 'national_avg_completion_rate'})
    .reset_index()
)

# Overall baselines
overall_hawaii   = compute_apprenticeship_metrics(active_data)
overall_national = compute_apprenticeship_metrics(active_nat)
hawaii_completion_rate   = overall_hawaii['completion_rate']
national_completion_rate = overall_national['completion_rate']

print(f"Hawaii programs:          {len(hi_metrics_by_program):,}")
print(f"National programs:        {len(national_metrics_by_program):,}")
print(f"Hawaii completion rate:   {hawaii_completion_rate:.1f}%")
print(f"National completion rate: {national_completion_rate:.1f}%")
print(f"Gap:                      {hawaii_completion_rate - national_completion_rate:+.1f}%")

Hawaii programs:          35
National programs:        12,905
Hawaii completion rate:   43.7%
National completion rate: 50.0%
Gap:                      -6.3%


In [14]:
# Apply MIN_N filter
filtered_hi = hi_metrics_by_program[
    hi_metrics_by_program['total_apprentices'] >= MIN_N
].copy()

# Then recompute filtered_national
filtered_national = national_metrics_by_program[
    national_metrics_by_program['total_apprentices'] >= MIN_N
].copy()

# Hawaii occ metrics with benchmark deviation
hi_occ_metrics = (
    active_data
    .groupby(['PROGRAM_ID', 'OCCUPATION_GROUP_NAME'])
    .apply(compute_apprenticeship_metrics)
    .reset_index()
    .merge(active_data[[
        'PROGRAM_ID','PROGRAM_NAME','USMAP_PROGRAM','FBOP_PROGRAM','NATIONAL_PROGRAM','UNION_Y_N',
        'SPON_FEDERALAGENCY','SPON_STATEAGENCY','SPON_CITYCOUNTYAGENCY','SPON_UNIONLABOR',
        'SPON_EMPLOYER','SPON_CMMNTYCLLGUNIV','SPON_WRKFRCDVLPMNTBRD','SPON_BSNSSASSCTN',
        'SPON_INTERMEDIARY','SPON_CMNTYBSDORG','SPON_FOUNDATION','SPON_OTHER',]]
        .drop_duplicates('PROGRAM_ID'), on='PROGRAM_ID', how='left')
    .merge(national_benchmark, on='OCCUPATION_GROUP_NAME', how='left')
)
hi_occ_metrics['deviation_from_national'] = (
    hi_occ_metrics['completion_rate'] - hi_occ_metrics['national_avg_completion_rate']
).round(1)
hi_occ_metrics = hi_occ_metrics[hi_occ_metrics['total_apprentices'] >= MIN_N].copy()

print(f"Hawaii programs (≥{MIN_N}):             {len(filtered_hi):,}")
print(f"National programs (≥{MIN_N}):           {len(filtered_national):,}")
print(f"Program × occupation combos (≥{MIN_N}): {len(hi_occ_metrics):,}")

Hawaii programs (≥20):             28
National programs (≥20):           2,751
Program × occupation combos (≥20): 38


## 6. Program Scale Visualization

In [16]:
hi_total  = len(hi_metrics_by_program)
nat_total = len(national_metrics_by_program)

min_n_values = np.arange(0, 100, 5)
hi_counts    = [(hi_metrics_by_program['total_apprentices'] >= n).sum() for n in min_n_values]
hi_pct       = [c / hi_total for c in hi_counts]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "Total Programs: Hawaiʻi vs. National",
        "Hawaiʻi Programs by Minimum Apprentice Threshold"
    ],
    specs=[[{"type": "bar"}, {"type": "xy"}]]
)

fig.add_trace(go.Bar(
    x=["National", "Hawaiʻi"], y=[nat_total, hi_total],
    text=[f"{nat_total:,}", f"{hi_total:,}"],
    textposition="inside", textfont=dict(color="white"),
    marker_color=["#1f77b4", "#2ca02c"],
    hovertemplate="<b>%{x}</b><br>Programs: %{y:,}<extra></extra>",
    showlegend=False,
), row=1, col=1)

fig.add_annotation(
    x=1, y=0.10, xref="x1", yref="paper",
    text=f"The national system is ~{nat_total // hi_total}× larger than Hawaiʻi's",
    showarrow=False,
    font=dict(size=11, color="#555555"),
    bgcolor="rgba(243,244,246,0.85)", bordercolor="#D1D5DB",
    borderwidth=1, borderpad=5
)

fig.add_trace(go.Bar(
    x=min_n_values, y=hi_counts,
    marker_color="#2ca02c",
    hovertemplate=(
        "<b>Minimum Apprentices: %{x}</b><br>"
        "Programs Remaining: %{y}<br>"
        "Share Retained: %{customdata:.1%}<extra></extra>"
    ),
    customdata=hi_pct, showlegend=False,
), row=1, col=2)

fig.add_annotation(
    x=.95, y=1, xref="x2 domain", yref="y2 domain",
    text="hover bars for details", showarrow=False,
    font=dict(size=10, color="#555555", style="italic"), xanchor="right",
)

fig.update_layout(
    title=f"{scope_label}: Program Scale and Filtering Effects",
    template="simple_white", showlegend=False, height=550,
)
fig.update_yaxes(
    type="log", title="Number of Programs (log scale)",
    tickvals=[1, 10, 100, 1000, 10000, 30000],
    ticktext=["1", "10", "100", "1,000", "10,000", "30,000"],
    row=1, col=1
)
fig.update_xaxes(title="Minimum Apprentices per Program", row=1, col=2)
fig.update_yaxes(title="Programs Remaining", row=1, col=2)
fig.show()

## 7. Threshold Analysis

Worst-case binomial margin of error: $SE_{\max} = \sqrt{\frac{0.25}{n}}$.

Observation-rate binomial margin of error: $SE = \sqrt{\frac{(\hat{p} * (1 - \hat{p}))}{n}}$.

Hawaiʻi's small program count means no threshold reaches the conventional ±10% 
margin of error. Estimates should be interpreted as directional rather than precise.


In [20]:
thresholds = np.arange(5, 155, 5)

df_thresholds = pd.DataFrame([{
    'MIN_N'                 : mn,
    'Programs_kept_hawaii'  : (hi_metrics_by_program['total_apprentices'] >= mn).sum(),
    'Programs_kept_national': (national_metrics_by_program['total_apprentices'] >= mn).sum(),
    'margin_of_error_pct'   : 1.96 * np.sqrt(0.25 / mn) * 100,
} for mn in thresholds])

df_thresholds['pct_hawaii'] = (
    df_thresholds['Programs_kept_hawaii'] / hi_total * 100
)

CUTOFF = SUPPRESSION_THRESHOLD

print(df_thresholds[['MIN_N', 'Programs_kept_hawaii', 'pct_hawaii', 'margin_of_error_pct']]
      .query('MIN_N <= 50').to_string(index=False))


 MIN_N  Programs_kept_hawaii  pct_hawaii  margin_of_error_pct
     5                    33   94.285714            43.826932
    10                    30   85.714286            30.990321
    15                    29   82.857143            25.303491
    20                    28   80.000000            21.913466
    25                    26   74.285714            19.600000
    30                    25   71.428571            17.892270
    35                    25   71.428571            16.565023
    40                    25   71.428571            15.495161
    45                    24   68.571429            14.608977
    50                    24   68.571429            13.859293


In [234]:
fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.55, 0.45],
    horizontal_spacing=0.05,
    specs=[[{"type": "xy", "secondary_y": True}, {"type": "table"}]],
    subplot_titles=["Coverage vs. Reliability Tradeoff", "Threshold Comparison"]
)

# --- Programs retained line (left y-axis) ---
fig.add_trace(go.Scatter(
    x=df_thresholds['MIN_N'],
    y=df_thresholds['pct_hawaii'],
    mode='lines+markers',
    name="Programs Retained",
    line=dict(color='#2ca02c', width=2.5),
    hovertemplate=(
        "<b>Minimum Apprentices: %{x}</b><br>"
        "Programs Retained: %{y:.1f}%<extra></extra>"
    ),
), row=1, col=1, secondary_y=False)

# --- Margin of error line (right y-axis) ---
fig.add_trace(go.Scatter(
    x=df_thresholds['MIN_N'],
    y=df_thresholds['margin_of_error_pct'],
    mode='lines',
    name="Margin of Error",
    line=dict(color='#d62728', width=2, dash='dot'),
    hovertemplate=(
        "<b>Minimum Apprentices: %{x}</b><br>"
        "Margin of Error: ±%{y:.1f}%<extra></extra>"
    ),
), row=1, col=1, secondary_y=True)

# --- Higher uncertainty zone ---
fig.add_vrect(
    x0=5, x1=MIN_N, row=1, col=1,
    fillcolor="rgba(255,100,100,0.06)", layer="below", line_width=0,
    annotation_text="Higher<br>uncertainty", annotation_position="bottom",
    annotation_font=dict(size=10, color="rgba(180,60,60,0.7)"),
)

# --- Cutoff line ---
fig.add_shape(
    type='line', x0=MIN_N, x1=MIN_N, y0=0, y1=1,
    xref='x', yref='paper',
    line=dict(color='grey', width=2, dash='dash'),
    row=1, col=1,
)

# --- Table ---
subset = df_thresholds.query('MIN_N <= 100').copy()
row_colors = [
    '#d4edda' if n == MIN_N else '#fde8e8' if n < MIN_N else 'white'
    for n in subset['MIN_N']
]

fig.add_trace(go.Table(
    header=dict(
        values=['<b>Min N</b>', '<b>Programs</b>', '<b>% Kept</b>', '<b>Margin of Error</b>'],
        fill_color='#f0f0f0', font=dict(size=11),
        align='center', line_color='lightgrey',
    ),
    cells=dict(
        values=[
            subset['MIN_N'], subset['Programs_kept_hawaii'],
            subset['pct_hawaii'].map('{:.1f}%'.format),
            subset['margin_of_error_pct'].map('±{:.1f}%'.format),
        ],
        fill_color=[row_colors] * 4,
        font=dict(size=11), align='center', line_color='lightgrey',
    )
), row=1, col=2)

# --- Axes ---
fig.update_xaxes(title='Minimum Apprentices per Program', range=[0, 100.5], row=1, col=1)
fig.update_yaxes(
    title='Programs Retained (%)', ticksuffix='%',
    color='#2ca02c', row=1, col=1, secondary_y=False
)
fig.update_yaxes(
    title='Margin of Error (%)', ticksuffix='%',
    color='#d62728', row=1, col=1, secondary_y=True
)

fig.update_layout(
    title=dict(text=(
        f"{scope_label}: Programs Retained vs. Data Reliability<br>"
        "<sup style='color:grey'>Green = programs retained · "
        "Red = margin of error · both improve as threshold increases</sup>"
    )),
    paper_bgcolor='white', plot_bgcolor='white',
    legend=dict(x=0.02, y=0.15, bgcolor='rgba(255,255,255,0.8)'),
    height=500,
    margin=dict(t=100, b=100, l=60, r=60),
)
fig.add_annotation(
    x=0.65, y=-0.18, xref='paper', yref='paper',
    text=(
        "* Margin of error based on worst-case 95% confidence interval (p=0.5).<br>"
        f"Hawaiʻi's limited total program count (n={hi_total}) means estimates exceed standard<br>"
        "±10% thresholds and should be interpreted as directional rather than precise."
    ),
    showarrow=False, font=dict(size=9, color='grey'),
    xanchor='left', align='right',
)

fig.update_yaxes(
    title='Percantage Rate (%)', ticksuffix='%',
    range=[0, 100],
    color='#2ca02c', row=1, col=1, secondary_y=False
)
fig.update_yaxes(
    range=[0, 100],
    showticklabels=False,   # hide redundant right axis labels
    showgrid=False,
    title='',
    row=1, col=1, secondary_y=True
)

fig.update_layout(
    legend=dict(
        orientation='h',      # horizontal
        x=0.5, xanchor='center',
        y=-0.15, yanchor='top',   # below x-axis
    ),
    margin=dict(t=100, b=130, l=60, r=30),  # extra bottom margin for legend
)
fig.show()

### 8. Funnel Plot

Compares every program against its own occupation's national benchmark,
standardized to a common scale so trades with very different baseline
difficulty are directly comparable on one chart.


In [21]:
TYPE_COLORS = {
    'Federal Agency Sponsor': '#1f77b4',          # blue
    'State/Local Government Sponsor': '#ff7f0e',  # orange
    'Union Labor Sponsor': '#d62728',             # red
    'Employer Sponsor': '#2ca02c',                # green
    'Education/Training Provider Sponsor': '#9467bd',  # purple
    'Association/Intermediary Sponsor': '#8c564b',     # brown
    'Other Sponsor Type': '#7f7f7f',              # gray
    'Unknown Sponsor Type': '#bcbd22'             # yellow‑green
}

def classify_primary_sponsor(row):
    # 1. Federal
    if row.get('SPON_FEDERALAGENCY') == 1:
        return 'Federal Agency Sponsor'
    
    # 2. State/Local Government
    if row.get('SPON_STATEAGENCY') == 1 or row.get('SPON_CITYCOUNTYAGENCY') == 1:
        return 'State/Local Government Sponsor'
    
    # 3. Union
    if row.get('SPON_UNIONLABOR') == 1:
        return 'Union Labor Sponsor'
    
    # 4. Employer
    if row.get('SPON_EMPLOYER') == 1:
        return 'Employer Sponsor'
    
    # 5. Education/Training Providers
    if row.get('SPON_CMMNTYCLLGUNIV') == 1:
        return 'Education/Training Provider Sponsor'
    if row.get('SPON_WRKFRCDVLPMNTBRD') == 1:   
        return 'Education/Training Provider Sponsor'
    
    # 6. Associations / Intermediaries / Nonprofits
    if row.get('SPON_BSNSSASSCTN') == 1:
        return 'Association/Intermediary Sponsor'
    if row.get('SPON_INTERMEDIARY') == 1:
        return 'Association/Intermediary Sponsor'
    if row.get('SPON_CMNTYBSDORG') == 1:        
        return 'Association/Intermediary Sponsor'
    if row.get('SPON_FOUNDATION') == 1:
        return 'Association/Intermediary Sponsor'
    
    # 7. Other
    if row.get('SPON_OTHER') == 1:
        return 'Other Sponsor Type'
    
    return 'Unknown Sponsor Type'



# Standardized residual: how many SDs each program's completion rate sits from
# ITS OWN occupation's national benchmark. This makes programs across different
# trades directly comparable on one chart — flat z-score axis instead of a raw
# completion-rate axis that would mix trades with very different baseline
# difficulty (e.g. HVAC vs Cement Masons).
p   = hi_occ_metrics['national_avg_completion_rate'] / 100
n   = hi_occ_metrics['total_apprentices']
se  = (np.sqrt(p * (1 - p) / n) * 100).replace(0, np.nan)
hi_occ_metrics['z_score']      = (hi_occ_metrics['completion_rate'] - hi_occ_metrics['national_avg_completion_rate']) / se
hi_occ_metrics['program_type'] = hi_occ_metrics.apply(classify_primary_sponsor, axis=1)

n_max = hi_occ_metrics['total_apprentices'].max()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=[MIN_N, n_max, n_max, MIN_N], y=[3, 3, -3, -3],
    fill='toself', fillcolor='rgba(255,200,200,0.25)',
    line=dict(color='rgba(0,0,0,0)'), name='±3 SD (~99.7%)', hoverinfo='skip',
))

fig.add_trace(go.Scatter(
    x=[MIN_N, n_max, n_max, MIN_N], y=[2, 2, -2, -2],
    fill='toself', fillcolor='rgba(200,230,200,0.35)',
    line=dict(color='rgba(0,0,0,0)'), name='±2 SD (~95.4%)', hoverinfo='skip',
))

for ptype, color in TYPE_COLORS.items():
    sub = hi_occ_metrics[hi_occ_metrics['program_type'] == ptype]
    if sub.empty:
        continue
    fig.add_trace(go.Scatter(
        x=sub['total_apprentices'], y=sub['z_score'],
        mode='markers', name=ptype,
        marker=dict(size=9, color=color, opacity=0.8),
        customdata=sub[['PROGRAM_NAME', 'OCCUPATION_GROUP_NAME',
                         'completion_rate', 'national_avg_completion_rate']].values,
        hovertemplate=(
            '<b>%{customdata[0]}</b><br>'
            'Occupation: %{customdata[1]}<br>'
            'Size: %{x}<br>'
            'Completion: %{customdata[2]:.1f}% (national: %{customdata[3]:.1f}%)<br>'
            'Standardized deviation: %{y:.1f} SD'
            '<extra></extra>'
        ),
    ))

fig.add_hline(y=0, line_dash='dash', line_color='grey',
              annotation_text="Matches national average for that program's trade")


fig.update_layout(
    title=(
        f'Funnel Plot: {scope_label} — Standardized Completion Rate vs Program Size<br>'
        "<sup>Y-axis = deviation from each program's own national occupation benchmark, in "
        "standard deviations — comparable across different trades. Points outside the band "
        "differ more than size alone would predict — a signal to investigate further, not a "
        "ranking.</sup>"
    ),
    xaxis_title='Total apprentices',
    yaxis_title='Standardized deviation from national benchmark (SD)',
    template='simple_white',
    font=dict(family='Inter, system-ui, Arial, sans-serif', size=13, color='#374151'),
    autosize=True, height=600,
)
fig.show()

## 9. Supply vs Demand Pipeline

- **Supply** = historical annual completions per occupation (RAPIDS)  
- **Demand lower** = `Openings_Change` — net new jobs from growth only  
- **Demand upper** = `Openings_Total` — all openings including replacements  

In [25]:
annual_supply = (
    active_data[
        (active_data['ACTUAL_COMPLETER'] == 1) &
        (active_data['OCCUPATION_GROUP_NAME'].notna())
    ]
    .groupby(['FISCAL_YEAR', 'OCCUPATION_GROUP_NAME'])
    .size()
    .reset_index(name='annual_completions')
)
avg_annual_supply = (
    annual_supply.groupby('OCCUPATION_GROUP_NAME')['annual_completions']
    .mean().round(2).reset_index(name='avg_annual_completions')
)

forecast_demand = (
    hawaii_merged[[
        'OCCUPATION_GROUP_NAME', 'Openings_Change', 'Openings_Total',
        'Annual_Growth', 'Employment_2024', 'Employment_2034', 'Median_Wage'
    ]]
    .dropna(subset=['Openings_Total', 'OCCUPATION_GROUP_NAME'])
    .groupby('OCCUPATION_GROUP_NAME', as_index=False).first()
)

supply_demand = avg_annual_supply.merge(forecast_demand, on='OCCUPATION_GROUP_NAME', how='inner')
supply_demand['gap_vs_lower'] = (supply_demand['avg_annual_completions'] - supply_demand['Openings_Change']).round(1)
supply_demand['gap_vs_upper'] = (supply_demand['avg_annual_completions'] - supply_demand['Openings_Total']).round(1)

print(f"Occupations with both supply and demand data: {len(supply_demand)}")
supply_demand[['OCCUPATION_GROUP_NAME', 'avg_annual_completions', 'Openings_Change', 'Openings_Total', 'gap_vs_lower', 'gap_vs_upper']].sort_values('gap_vs_upper')

Occupations with both supply and demand data: 20


,OCCUPATION_GROUP_NAME,avg_annual_completions,Openings_Change,Openings_Total,gap_vs_lower,gap_vs_upper
1,Carpenters,43.25,40.0,640.0,3.2,-596.8
3,Construction Laborers,41.75,50.0,540.0,-8.2,-498.2
6,Electricians,31.25,40.0,340.0,-8.8,-308.8
13,"Plumbers, Pipefitters, and Steamfitters",32.92,10.0,260.0,22.9,-227.1
11,"Painters, Construction and Maintenance",16.08,10.0,210.0,6.1,-193.9
10,Operating Engineers and Other Construction Equ...,8.67,10.0,160.0,-1.3,-151.3
9,"Heating, Air Conditioning, and Refrigeration M...",20.50,10.0,110.0,10.5,-89.5
14,Roofers,1.00,10.0,90.0,-9.0,-89.0
4,Drywall and Ceiling Tile Installers,10.11,10.0,80.0,0.1,-69.9
18,Telecommunications Line Installers and Repairers,1.00,0.0,70.0,1.0,-69.0


In [26]:
# ── Compute gap interval ───────────────────────────────────────────────────
supply_demand_plot = supply_demand.copy()

supply_demand_plot['gap_worst'] = (          # supply − total openings (worst case)
    supply_demand_plot['avg_annual_completions'] -
    supply_demand_plot['Openings_Total']
).round(0)

supply_demand_plot['gap_best'] = (           # supply − new jobs only (best case)
    supply_demand_plot['avg_annual_completions'] -
    supply_demand_plot['Openings_Change'].clip(lower=0)  # clamp negative growth
).round(0)

# Sort least→greatest shortage (least negative at top, most negative at bottom)
supply_demand_plot = supply_demand_plot.sort_values('gap_worst', ascending=True)

# ── Figure ─────────────────────────────────────────────────────────────────
fig = go.Figure()

# Range bars — coloured by whether fully in shortfall, surplus, or mixed
for _, row in supply_demand_plot.iterrows():
    lo, hi = row['gap_worst'], row['gap_best']
    if hi <= 0:
        color = 'rgba(185,28,28,0.35)'    # entirely in shortfall
    elif lo >= 0:
        color = 'rgba(21,128,61,0.35)'    # entirely in surplus
    else:
        color = 'rgba(217,119,6,0.35)'    # range crosses zero (mixed)

    if abs(hi - lo) > 0.5:               # skip invisible ranges
        fig.add_trace(go.Scatter(
            x=[lo, hi],
            y=[row['OCCUPATION_GROUP_NAME'], row['OCCUPATION_GROUP_NAME']],
            mode='lines', line=dict(color=color, width=8),
            showlegend=False, hoverinfo='skip',
        ))

# Worst-case gap markers (left endpoint)
fig.add_trace(go.Scatter(
    x=supply_demand_plot['gap_worst'],
    y=supply_demand_plot['OCCUPATION_GROUP_NAME'],
    mode='markers', name='Gap vs all openings (worst case)',
    marker=dict(color='#B91C1C', size=10, symbol='triangle-left'),
    hovertemplate='%{y}<br>Gap vs all openings: %{x:+.0f} workers/yr<extra></extra>',
))

# Best-case gap markers (right endpoint)
fig.add_trace(go.Scatter(
    x=supply_demand_plot['gap_best'],
    y=supply_demand_plot['OCCUPATION_GROUP_NAME'],
    mode='markers', name='Gap vs new jobs only (best case)',
    marker=dict(color='#D97706', size=10, symbol='triangle-right'),
    hovertemplate='%{y}<br>Gap vs new jobs only: %{x:+.0f} workers/yr<extra></extra>',
))

fig.update_layout(
    autosize=True,
    title=dict(
        text=(
            f'<b>Apprenticeship Supply vs Labor Demand — {scope_label}</b><br>'
            '<span style="font-size:13px;color:#6B7280">'
            'Values = avg annual graduates minus annual demand. '
            'Left of zero = shortfall. '
            'Range = uncertainty between growth-only vs all-openings demand. '
            'Shaded Row = ■ BLS "Apprenticeship" training category</span>'
        ),
        font=dict(size=18, family='Inter, system-ui, Arial, sans-serif'),
        x=0.0, xanchor='left',
    ),
    xaxis=dict(
        title='Annual surplus (+) / shortage (−)  [workers per year]',
        zeroline=True, zerolinewidth=2, zerolinecolor='rgba(0,0,0,0.35)',
        tickformat='+d',
    ),
    yaxis_title='Occupation',
    template='simple_white',
    font=dict(family='Inter, system-ui, Arial, sans-serif', size=13, color='#374151'),
    height=max(500, len(supply_demand_plot) * 40),
    margin=dict(l=350, r=60, t=130, b=80),
    legend=dict(
        x=.02, y=0.90, borderwidth=0.5,
        font=dict(size=12, family='Inter, system-ui, Arial, sans-serif'),
    ),
)

fig.update_layout(
    legend=dict(
        orientation='h',
        x=0, xanchor='left',
        y=1, yanchor='bottom',   # sits just above the plot, below the title block
        bgcolor='rgba(255,255,255,0)',
        font=dict(size=12, family='Inter, system-ui, Arial, sans-serif'),
    ),
)

# BLS Employment Projections — fixed national "Apprenticeship" training-category occupations.
# Confirmed against Hawaii's own Long-Term Occupational Forecast "Job Training" field.
requires_apprenticeship = {
    'Brickmasons and Blockmasons',
    'Carpenters',
    'Electricians',
    'Glaziers',
    'Insulation Workers, Mechanical',   # add to dataset if currently dropped by MIN_N
    'Plumbers, Pipefitters, and Steamfitters',
    'Sheet Metal Workers',
    'Structural Iron and Steel Workers',
}

color_required = 'rgba(22, 132, 65, 0.18)'  # single green tier — no second color needed

category_order = supply_demand_plot['OCCUPATION_GROUP_NAME'].tolist()
fig.update_yaxes(categoryorder='array', categoryarray=category_order)

for name in category_order:
    if name not in requires_apprenticeship:
        continue
    i = category_order.index(name)
    fig.add_shape(
        type='rect', xref='paper', x0=0, x1=1, yref='y', y0=i - 0.5, y1=i + 0.5,
        fillcolor=color_required, line_width=0, layer='below',
    )

ticktext = [f'<b>{c}</b>' if c in requires_apprenticeship else c for c in category_order]
fig.update_yaxes(tickmode='array', tickvals=category_order, ticktext=ticktext)

fig.show(config={'responsive': True})


In [27]:
summary = active_data.groupby('OCCUPATION_GROUP_NAME').agg(
    total_apprentices=('APPRENTICE_STATUS_CODE', 'size'),
    completion_rate=('APPRENTICE_STATUS_CODE', lambda x: (x == 'Completed').mean()),
)

summary['completion_rate'] = (summary['completion_rate'] * 100).round(1)
summary = summary.sort_values('total_apprentices', ascending=False)
summary

,total_apprentices,completion_rate
OCCUPATION_GROUP_NAME,,
Carpenters,2684,36.1
Electricians,1378,42.0
Roofers,943,5.4
"Plumbers, Pipefitters, and Steamfitters",895,67.4
"Painters, Construction and Maintenance",890,41.0
Construction Laborers,723,74.1
Drywall and Ceiling Tile Installers,653,29.4
Tile and Stone Setters,449,7.8
"Heating, Air Conditioning, and Refrigeration Mechanics and Installers",358,86.9


## 10. Interactive Dashboard

The dashboard below has a pipeline (Sankey), trend,
enrollment, and demographics views for any occupation and program.

It also includes a clickable version
of the standardized funnel plot from previous code so clicking a flagged program
jumps straight to its full breakdown.


In [28]:
# ── Occupation + Program filter (with disclosure suppression) ──────────────
# FERPA-style minimum cell size: small occupations/programs are never listed
# individually. Where 2+ small groups exist they're folded into a single
# "(combined)" option so totals still reconcile without exposing any one
# small group's exact count (complementary suppression).

def get_occupation_options():
    counts     = active_data['OCCUPATION_GROUP_NAME'].value_counts()
    safe_occs  = sorted(counts[counts >= SUPPRESSION_THRESHOLD].index.tolist())
    small_occs = counts[counts < SUPPRESSION_THRESHOLD]
    options = ['All Trades'] + safe_occs
    if len(small_occs) >= 2:
        options.append('Other Occupations (combined)')
    return options

def _occupation_subset(occ):
    if occ == 'All Trades':
        return active_data
    if occ == 'Other Occupations (combined)':
        counts     = active_data['OCCUPATION_GROUP_NAME'].value_counts()
        small_occs = counts[counts < SUPPRESSION_THRESHOLD].index
        return active_data[active_data['OCCUPATION_GROUP_NAME'].isin(small_occs)]
    return active_data[active_data['OCCUPATION_GROUP_NAME'] == occ]

def get_programs_for_occ(occ):
    base   = _occupation_subset(occ)
    counts = base['PROGRAM_NAME'].value_counts()
    safe_programs  = sorted(counts[counts >= SUPPRESSION_THRESHOLD].index.tolist())
    small_programs = counts[counts < SUPPRESSION_THRESHOLD]
    options = ['All Programs'] + safe_programs
    # Only offer the combined bucket when there's at least one named safe
    # program alongside it — otherwise it's identical to 'All Programs'.
    if safe_programs and len(small_programs) >= 2:
        options.append('Other Programs (combined)')
    return options

def get_filtered(occ, program='All Programs'):
    df = _occupation_subset(occ)
    if program == 'Other Programs (combined)':
        counts         = df['PROGRAM_NAME'].value_counts()
        small_programs = counts[counts < SUPPRESSION_THRESHOLD].index
        df = df[df['PROGRAM_NAME'].isin(small_programs)]
    elif program != 'All Programs':
        df = df[df['PROGRAM_NAME'] == program]
    return df

def is_suppressed(df):
    return len(df) < SUPPRESSION_THRESHOLD


In [29]:
# ── Palette ──────────────────────────────────────────────────────────────
NODE_COLORS = ['#1E40AF','#15803D','#B91C1C','#475569','#C2410C','#D97706','#94A3B8']
LINK_COLORS = [
    'rgba(21,128,61,0.22)', 'rgba(185,28,28,0.22)', 'rgba(71,85,105,0.15)',
    'rgba(194,65,12,0.25)', 'rgba(217,119,6,0.25)', 'rgba(148,163,184,0.18)',
]

def _pct(n, total):
    return f'{n / total * 100:.1f}%' if total > 0 else '—'

def _label(name, n, total):
    return f'<b>{name}</b><br>{n:,}<br><span style="opacity:0.6">{_pct(n, total)}</span>'

def _compute(df):
    total        = len(df)
    completed    = (df['APPRENTICE_STATUS_CODE'] == 'Completed').sum()
    cancelled    = (df['APPRENTICE_STATUS_CODE'] == 'Cancelled').sum()
    active_ct    = (df['APPRENTICE_STATUS_CODE'] == 'Active').sum()
    early_exit   = (df['EARLY_TRAINING_EXIT_TYPE'] == 1).sum()
    late_exit    = (df['EARLY_TRAINING_EXIT_TYPE'] == 2).sum()
    unknown_exit = max(cancelled - early_exit - late_exit, 0)
    return total, completed, cancelled, active_ct, early_exit, late_exit, unknown_exit

def _node_positions(total, completed, cancelled, active_ct, early, late, unknown):
    PAD = 0.02
    right_stack = [
        (completed, 1),
        (active_ct, 3),
        (unknown,   6),
        (late,      5),
        (early,     4),
    ]
    visible = [(v, i) for v, i in right_stack if v > 0]
    n       = len(visible)
    scale   = max(1.0 - PAD * max(n - 1, 0), 0.5)

    pos = {}
    cum = 0.0
    for val, idx in visible:
        h = (val / total) * scale
        pos[idx] = (0.99, cum + h / 2)
        cum += h + PAD

    if cancelled > 0:
        subs = [(pos[i][1], v) for v, i in visible if i in {4, 5, 6}]
        if subs:
            tot_sub = sum(v for _, v in subs)
            pos[2] = (0.60, sum(y * v for y, v in subs) / tot_sub)
        else:
            pos[2] = (0.60, (completed + active_ct + cancelled / 2) / total)

    ys     = [y for _, y in pos.values()]
    pos[0] = (0.01, (min(ys) + max(ys)) / 2)

    y_vals = [y for _, (_, y) in pos.items()]
    lo, hi = min(y_vals), max(y_vals)

    if (hi - lo) > 0.87:
        mid = (lo + hi) / 2
        pos = {i: (x, 0.5 + (y - mid) * (0.87 / (hi - lo)))
               for i, (x, y) in pos.items()}
        y_vals = [y for _, (_, y) in pos.items()]
        lo, hi = min(y_vals), max(y_vals)

    if hi > 0.92:
        d = hi - 0.92
        pos = {i: (x, y - d) for i, (x, y) in pos.items()}
    elif lo < 0.05:
        d = 0.05 - lo
        pos = {i: (x, y + d) for i, (x, y) in pos.items()}

    return pos

def _active_sankey_data(df):
    total, completed, cancelled, active_ct, early, late, unknown = _compute(df)
    NODE_REG = [
        ('Enrolled',            total,     NODE_COLORS[0]),
        ('Completed',           completed, NODE_COLORS[1]),
        ('Cancelled',           cancelled, NODE_COLORS[2]),
        ('Still Active',        active_ct, NODE_COLORS[3]),
        ('Early Exit',          early,     NODE_COLORS[4]),
        ('Late Exit',           late,      NODE_COLORS[5]),
        ('Unknown Exit Timing', unknown,   NODE_COLORS[6]),
    ]
    LINK_REG = [
        (0, 1, completed, LINK_COLORS[0], 'Completed'),
        (0, 2, cancelled, LINK_COLORS[1], 'Cancelled'),
        (0, 3, active_ct, LINK_COLORS[2], 'Still Active'),
        (2, 4, early,     LINK_COLORS[3], 'Early Exit'),
        (2, 5, late,      LINK_COLORS[4], 'Late Exit'),
        (2, 6, unknown,   LINK_COLORS[5], 'Unknown Exit Timing'),
    ]
    live = [(s, t, v, c, n) for s, t, v, c, n in LINK_REG if v > 0]

    used_idx = {0}
    for s, t, *_ in live:
        used_idx.add(s); used_idx.add(t)

    remap       = {old: new for new, old in enumerate(sorted(used_idx))}
    pos         = _node_positions(total, completed, cancelled, active_ct, early, late, unknown)
    sorted_used = sorted(used_idx)

    return dict(
        labels  = [_label(NODE_REG[i][0], NODE_REG[i][1], total) for i in sorted_used],
        colors  = [NODE_REG[i][2]                                 for i in sorted_used],
        node_x  = [pos.get(i, (0.5, 0.5))[0]                     for i in sorted_used],
        node_y  = [pos.get(i, (0.5, 0.5))[1]                     for i in sorted_used],
        sources = [remap[s]            for s, *_ in live],
        targets = [remap[t]            for _, t, *_ in live],
        values  = [v                   for _, _, v, *_ in live],
        lcolors = [c                   for _, _, _, c, _ in live],
        cdata   = [[_pct(v, total), n] for _, _, v, _, n in live],
        total   = total,
    )

def _build_sankey_fig(df, occ):
    d = _active_sankey_data(df)
    fig = go.Figure(go.Sankey(
        arrangement='fixed',
        node=dict(
            label=d['labels'], color=d['colors'],
            x=d['node_x'],     y=d['node_y'],
            pad=32, thickness=18,
            line=dict(color='white', width=1),
            hovertemplate='<b>%{label}</b><extra></extra>',
        ),
        link=dict(
            source=d['sources'], target=d['targets'],
            value=d['values'],   color=d['lcolors'],
            customdata=d['cdata'],
            hovertemplate=(
                '<b>%{customdata[1]}</b><br>'
                'Apprentices: %{value:,}<br>'
                'Share of enrolled: %{customdata[0]}<extra></extra>'
            ),
        ),
        textfont=dict(size=11, color='#374151',
                      family='Inter, system-ui, Arial, sans-serif'),
    ))
    fig.update_layout(
        title=dict(
            text=(f'<b>{scope_label}: Apprenticeship Pipeline</b>'
                  f'<br><span style="font-size:13px;color:#6B7280">'
                  f'{occ} · {d["total"]:,} total apprentices in all fiscal years combined</span>'),
            font=dict(size=18, family='Inter, system-ui, Arial, sans-serif'),
            x=0.0, xanchor='left', y=0.97,
        ),
        font=dict(family='Inter, system-ui, Arial, sans-serif', size=13, color='#374151'),
        template='simple_white',
        autosize=True,
        height=720,
        margin=dict(t=110, b=80, l=20, r=200),
    )
    return fig


In [30]:
def compute_exit_trend(df, min_resolved=CUTOFF):
    trend = (
        df.groupby('FISCAL_YEAR')
        .agg(
            exits_during  = ('EARLY_TRAINING_EXIT_TYPE', lambda x: (x == 1).sum()),
            cancellations = ('CANCELLED_APPR', 'sum'),
            completions   = ('ACTUAL_COMPLETER', 'sum'),
        )
        .reset_index()
    )
    n = trend['completions'] + trend['cancellations']

    mask  = n >= min_resolved
    trend = trend[mask].copy().reset_index(drop=True)
    n     = n[mask].reset_index(drop=True)

    if trend.empty:
        return trend

    p_comp = trend['completions']   / n
    p_canc = trend['cancellations'] / n

    trend['completion_rate']  = (p_comp * 100).round(1)
    trend['early_exit_lower'] = (trend['exits_during'] / n * 100).round(1)
    trend['early_exit_upper'] = (p_canc * 100).round(1)
    trend['n']                = n.values

    # 95% Wald confidence intervals
    z           = 1.96
    comp_margin = z * np.sqrt(p_comp * (1 - p_comp) / n) * 100
    canc_margin = z * np.sqrt(p_canc * (1 - p_canc) / n) * 100

    trend['comp_lo'] = (trend['completion_rate']  - comp_margin).clip(0, 100).round(1)
    trend['comp_hi'] = (trend['completion_rate']  + comp_margin).clip(0, 100).round(1)
    trend['canc_lo'] = (trend['early_exit_upper'] - canc_margin).clip(0, 100).round(1)
    trend['canc_hi'] = (trend['early_exit_upper'] + canc_margin).clip(0, 100).round(1)

    return trend


def _empty_state_fig(title_text):
    fig = go.Figure()
    fig.add_annotation(
        text='No apprentices match this filter — count below the minimum reporting threshold',
        x=0.5, y=0.5, xref='paper', yref='paper',
        showarrow=False, align='center',
        font=dict(size=13, color='#6B7280', family='Inter, system-ui, Arial, sans-serif'),
    )
    fig.update_layout(
        title=title_text, template='simple_white', height=300,
        xaxis=dict(visible=False), yaxis=dict(visible=False),
        autosize=True,
    )
    return fig


def _build_trend_fig(df, occ):
    trend = compute_exit_trend(df, min_resolved=CUTOFF)

    if trend.empty or len(trend) < 2:
        moe = df_thresholds.loc[
            df_thresholds['MIN_N'] == CUTOFF, 'margin_of_error_pct'
        ].values[0]
        fig = go.Figure()
        fig.add_annotation(
            text=(
                f'<b>Insufficient cohort data</b><br>'
                f'Fewer than 2 fiscal years meet the minimum of {CUTOFF} resolved apprentices<br>'
                f'<span style="color:#9CA3AF">'
                f'(threshold set to keep margin of error within ±{moe:.1f} pp)</span>'
            ),
            x=0.5, y=0.5, xref='paper', yref='paper',
            showarrow=False, align='center',
            font=dict(size=13, color='#6B7280',
                      family='Inter, system-ui, Arial, sans-serif'),
        )
        fig.update_layout(
            title=(f'<b>{scope_label}: Completion vs. Cancellation Rate</b>'
                   f'<br><span style="font-size:12px;color:#6B7280">'
                   f'{occ} · Insufficient data for trend analysis</span>'),
            template='simple_white', height=450, autosize=True,
            margin=dict(t=100, b=90),
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
        )
        return fig

    fig = go.Figure()
    for hi_col, lo_col, fill_color in [
        ('comp_hi', 'comp_lo', 'rgba(21,128,61,0.13)'),
        ('canc_hi', 'canc_lo', 'rgba(185,28,28,0.13)'),
    ]:
        fig.add_trace(go.Scatter(
            x=trend['FISCAL_YEAR'], y=trend[hi_col],
            mode='lines', line=dict(width=0),
            showlegend=False, hoverinfo='skip',
        ))
        fig.add_trace(go.Scatter(
            x=trend['FISCAL_YEAR'], y=trend[lo_col],
            mode='lines', fill='tonexty', fillcolor=fill_color,
            line=dict(width=0),
            showlegend=False, hoverinfo='skip',
        ))

    fig.add_trace(go.Scatter(
        x=trend['FISCAL_YEAR'], y=trend['completion_rate'],
        mode='lines+markers', name='Completion Rate',
        line=dict(color='#15803D', width=2.5),
        customdata=trend[['n', 'comp_lo', 'comp_hi']].values,
        hovertemplate=(
            'FY%{x}<br>'
            'Completion: %{y:.1f}%<br>'
            '95%% CI: %{customdata[1]:.1f}%–%{customdata[2]:.1f}%<br>'
            'Resolved n = %{customdata[0]:,.0f}'
            '<extra></extra>'
        ),
    ))
    fig.add_trace(go.Scatter(
        x=trend['FISCAL_YEAR'], y=trend['early_exit_upper'],
        mode='lines+markers', name='Cancellation Rate',
        line=dict(color='#B91C1C', width=2.5),
        customdata=trend[['n', 'canc_lo', 'canc_hi']].values,
        hovertemplate=(
            'FY%{x}<br>'
            'Cancellation: %{y:.1f}%<br>'
            '95%% CI: %{customdata[1]:.1f}%–%{customdata[2]:.1f}%<br>'
            'Resolved n = %{customdata[0]:,.0f}'
            '<extra></extra>'
        ),
    ))
    fig.update_layout(
        title=(f'<b>{scope_label}: Completion vs. Cancellation Rate</b>'
               f'<br><span style="font-size:12px;color:#6B7280">'
               f'{occ} · By fiscal year of record (not enrollment year) · Resolved outcomes only · Shaded bands = 95% CI'
        ),
        xaxis=dict(title='Fiscal Year', tickformat='d'),
        yaxis=dict(title='Rate (%)', ticksuffix='%', range=[0, 105]),
        template='simple_white',
        font=dict(family='Inter, system-ui, Arial, sans-serif', size=13),
        legend=dict(orientation='h', x=0.5, xanchor='center', y=-0.18),
        autosize=True,
        height=450, margin=dict(t=100, b=90),
    )
    return fig


In [31]:
def _build_enrollment_fig(df, occ, recent_years=3):
    enrol = (
        df.groupby('FISCAL_YEAR')
          .size()
          .reset_index(name='new_enrollments')
    )
    enrol = enrol[enrol['new_enrollments'] >= SUPPRESSION_THRESHOLD].copy()

    if enrol.empty:
        return _empty_state_fig(
            f'<b>Annual Enrollment & Projected Graduates</b>'
            f'<br><span style="font-size:12px;color:#6B7280">'
            f'{occ} · No fiscal year meets the minimum reporting threshold</span>'
        )

    resolved    = df['APPRENTICE_STATUS_CODE'].isin(['Completed', 'Cancelled'])
    hist_rate   = (df['APPRENTICE_STATUS_CODE'] == 'Completed').sum() / resolved.sum()

    # Recent rate: last N years of resolved outcomes only
    max_yr      = int(df['FISCAL_YEAR'].max())
    recent_df   = df[df['FISCAL_YEAR'] >= max_yr - recent_years]
    res_recent  = recent_df['APPRENTICE_STATUS_CODE'].isin(['Completed', 'Cancelled']).sum()
    recent_rate = (
        (recent_df['APPRENTICE_STATUS_CODE'] == 'Completed').sum() / res_recent
        if res_recent >= SUPPRESSION_THRESHOLD else None
    )

    enrol['proj_hist'] = (enrol['new_enrollments'] * hist_rate).round(1)
    if recent_rate is not None:
        enrol['proj_recent'] = (enrol['new_enrollments'] * recent_rate).round(1)

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=enrol['FISCAL_YEAR'], y=enrol['new_enrollments'],
        name='New enrollments',
        marker=dict(color='#fef0f8', line=dict(color='#c550a3', width=2)),
        hovertemplate='FY%{x}<br>New enrollments: %{y:,}<extra></extra>',
    ))

    # Conservative projection — historical average. Lags the recent-trend
    # line by design: completions from new enrollees won't show up for
    # several years, so the historical rate is still the valid estimate
    # for cohorts currently mid-program.
    fig.add_trace(go.Scatter(
        x=enrol['FISCAL_YEAR'], y=enrol['proj_hist'],
        mode='lines+markers', name=f'Projected — historical ({hist_rate*100:.1f}%)',
        line=dict(color='#415262', width=2, dash='dot'),
        hovertemplate=(
            f'FY%{{x}}<br>Projected graduates: %{{y:.0f}}<br>'
            f'Based on {hist_rate*100:.1f}% historical rate'
            '<extra></extra>'
        ),
    ))

    # Optimistic projection — recent trend, reflecting what NEW entrants
    # are likely experiencing under whatever has recently improved.
    if recent_rate is not None:
        fig.add_trace(go.Scatter(
            x=enrol['FISCAL_YEAR'], y=enrol['proj_recent'],
            mode='lines+markers', name=f'Projected — recent {recent_years}-yr ({recent_rate*100:.1f}%)',
            line=dict(color='#15803D', width=2.5, dash='dash'),
            hovertemplate=(
                f'FY%{{x}}<br>Projected graduates: %{{y:.0f}}<br>'
                f'Based on {recent_rate*100:.1f}% rate (last {recent_years} yrs)'
                '<extra></extra>'
            ),
        ))

    subtitle = (
        f'{occ} · Historical line reflects current mid-program cohorts · '
        f'Recent line reflects new entrants experiencing the current program'
    )

    fig.update_layout(
        title=(f'<b>Annual Enrollment & Projected Graduates</b><br>'
               f'<span style="font-size:12px;color:#6B7280">{subtitle}</span>'),
        xaxis=dict(title='Fiscal Year', tickformat='d'),
        yaxis=dict(title='Apprentices'),
        template='simple_white',
        font=dict(family='Inter, system-ui, Arial, sans-serif', size=13),
        legend=dict(orientation='h', x=0.5, xanchor='center', y=-0.28),
        barmode='overlay',
        autosize=True,
        height=420, margin=dict(t=110, b=110),
    )
    return fig


In [32]:
BAND_FILL = {'Completed': '#15803d26', 'Cancelled': '#b91c1c26'}

# Stated assumption, not derived from program-level payroll data — actual
# apprentice wages vary widely (commonly $20-24+/hr) and differ by program,
# and this dataset's Median_Wage column reflects journey-level annual pay
# (BLS convention), not a per-apprentice hourly rate, so it can't be used
# directly without risking exactly this kind of unit error.
ASSUMED_APPRENTICE_WAGE = 20    # $/hour
ASSUMED_HOURS_PER_YEAR  = 2080  # 40 hrs/week × 52 weeks

def estimate_sunk_cost(years_elapsed):
    """
    Illustrative training-wage figure under one flat, explicitly stated
    assumption (see ASSUMED_APPRENTICE_WAGE / ASSUMED_HOURS_PER_YEAR above).
    Not a per-program or per-cohort estimate — every panel uses the same
    assumption, and the annotation states it plainly each time it appears.
    """
    return ASSUMED_APPRENTICE_WAGE * ASSUMED_HOURS_PER_YEAR * years_elapsed


def build_program_survival_fig(df, label, nat_df=None):
    d = df.copy()
    d['START_DATE']   = pd.to_datetime(d['START_DATE'])
    exited            = d['APPRENTICE_STATUS_CODE'].isin(['Completed', 'Cancelled'])
    censored_days     = (DATA_AS_OF - d['START_DATE']).dt.days
    raw_days          = d['duration_days'].where(exited, censored_days)
    d['duration_y']   = raw_days / 365.25
    d = d[d['duration_y'].notna() & (d['duration_y'] > 0)].copy()

    conditions     = [d['APPRENTICE_STATUS_CODE'] == 'Completed',
                      d['APPRENTICE_STATUS_CODE'] == 'Cancelled']
    d['event']     = np.select(conditions, [1, 2], default=0)
    d['any_event'] = (d['event'] != 0).astype(int)

    if len(d) < SUPPRESSION_THRESHOLD:
        return _empty_state_fig(f'<b>{label}</b>')

    # ── Cohort split: auto-detect where recent enrollment begins ───────────
    split_yr   = 0
    hist_sub   = d.iloc[:0]
    recent_sub = d.iloc[:0]
    fiscal_years = sorted(d['FISCAL_YEAR'].dropna().unique())
    if len(fiscal_years) >= 6:
        split_yr    = int(fiscal_years[-len(fiscal_years) // 3])
        d['cohort'] = np.where(
            d['FISCAL_YEAR'] >= split_yr,
            f'Recent (≥{split_yr})',
            f'Historical (<{split_yr})',
        )
        show_cohort_split = True
    else:
        show_cohort_split = False

    total_n        = len(d)
    expected_years = d['YEARS_TO_COMPLETE'].median()
    x_max          = float(expected_years) * 1.3 if pd.notna(expected_years) else 8.0

    pct_cancel_yr1  = ((d['event'] == 2) & (d['duration_y'] <= 1.0)).sum() / total_n * 100
    cancelled_times = d.loc[d['event'] == 2, 'duration_y']
    median_cancel_y = (cancelled_times.median()
                        if len(cancelled_times) >= SUPPRESSION_THRESHOLD else None)

    def _add_panel_traces(fig, sub, row, col, show_legend):
        sub_n = len(sub)
        risk_grid = np.linspace(0, x_max, 200)
        at_risk   = np.array([(sub['duration_y'] >= tt).sum() for tt in risk_grid])
        valid     = at_risk >= SUPPRESSION_THRESHOLD
        t_cutoff  = risk_grid[valid].max() if valid.any() else 0.0

        for event_code, name, color in [(1, 'Completed', '#15803D'),
                                         (2, 'Cancelled', '#B91C1C')]:
            try:
                ajf = AalenJohansenFitter()
                ajf.fit(sub['duration_y'], sub['event'], event_of_interest=event_code)
                cif  = ajf.cumulative_density_
                t    = cif.index.to_numpy(dtype=float)
                pct  = cif.iloc[:, 0].to_numpy(dtype=float) * 100
                cnt  = cif.iloc[:, 0].to_numpy(dtype=float) * sub_n
                mask = (t <= x_max) & (t <= t_cutoff)

                try:
                    ci = ajf.confidence_interval_
                    if ci is None:
                        raise ValueError
                    lo = ci.iloc[:, 0].to_numpy(dtype=float)[mask] * 100
                    hi = ci.iloc[:, 1].to_numpy(dtype=float)[mask] * 100
                    fig.add_trace(go.Scatter(
                        x=t[mask], y=hi, mode='lines', line=dict(width=0),
                        showlegend=False, hoverinfo='skip',
                    ), row=row, col=col)
                    fig.add_trace(go.Scatter(
                        x=t[mask], y=lo, mode='lines', fill='tonexty',
                        fillcolor=BAND_FILL[name], line=dict(width=0),
                        showlegend=False, hoverinfo='skip',
                    ), row=row, col=col)
                except Exception:
                    pass

                fig.add_trace(go.Scatter(
                    x=t[mask], y=pct[mask], mode='lines',
                    line=dict(color=color, width=2.5, shape='hv'),
                    name=name, showlegend=show_legend,
                    customdata=cnt[mask],
                    hovertemplate=(f'<b>{name}</b><br>Year %{{x:.1f}}<br>'
                                   f'%{{y:.1f}}% (%{{customdata:.0f}} of {sub_n:,})'
                                   f'<extra></extra>'),
                ), row=row, col=col)
            except Exception:
                pass

        try:
            kmf = KaplanMeierFitter()
            kmf.fit(sub['duration_y'], event_observed=sub['any_event'])
            sf     = kmf.survival_function_
            t_tot  = sf.index.to_numpy(dtype=float)
            y_tot  = (1 - sf.iloc[:, 0].to_numpy(dtype=float)) * 100
            mask_t = (t_tot <= x_max) & (t_tot <= t_cutoff)
            fig.add_trace(go.Scatter(
                x=t_tot[mask_t], y=y_tot[mask_t], mode='lines',
                name='Total resolved', showlegend=show_legend,
                line=dict(color='#475569', width=2, dash='dash', shape='hv'),
                hovertemplate=('<b>Total resolved</b><br>Year %{x:.1f}<br>'
                               '%{y:.1f}% — remainder still enrolled<extra></extra>'),
            ), row=row, col=col)
        except Exception:
            pass

    # ── Build figure ─────────────────────────────────────────────────────────
    if show_cohort_split:
        hist_sub   = d[d['cohort'] == f'Historical (<{split_yr})']
        recent_sub = d[d['cohort'] == f'Recent (≥{split_yr})']
        has_hist   = len(hist_sub) >= SUPPRESSION_THRESHOLD
        has_recent = len(recent_sub) >= SUPPRESSION_THRESHOLD

        if has_hist and has_recent:
            fig = make_subplots(
                rows=1, cols=2,
                specs=[[{"secondary_y": False}, {"secondary_y": False}]],
                subplot_titles=[
                    f'Before {split_yr}  (n={len(hist_sub):,})',
                    f'{split_yr} or later  (n={len(recent_sub):,})',
                ],
                shared_yaxes=True,
                horizontal_spacing=0.06,
            )
            _add_panel_traces(fig, hist_sub,   row=1, col=1, show_legend=True)
            _add_panel_traces(fig, recent_sub, row=1, col=2, show_legend=False)
        else:
            fig = go.Figure()
            _add_panel_traces(fig, d, row=None, col=None, show_legend=True)
            show_cohort_split = False
    else:
        fig = go.Figure()
        _add_panel_traces(fig, d, row=None, col=None, show_legend=True)

    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='lines',
        line=dict(color='#B91C1C', width=1.2, dash='dash'),
        opacity=0.5, name='Median cancellation point',
    ))

    # ── Per-column medians, computed up front so the recent panel's box can
    #    reference the historical panel's median for the comparison line ────
    n_cols = 2 if show_cohort_split else 1

    if show_cohort_split:
        cohort_medians, pct_by_yr1_col = {}, {}
        for cname, sub in [(f'Historical (<{split_yr})', hist_sub),
                            (f'Recent (≥{split_yr})',    recent_sub)]:
            ct = sub.loc[sub['event'] == 2, 'duration_y']
            cohort_medians[cname] = ct.median() if len(ct) >= SUPPRESSION_THRESHOLD else None
            pct_by_yr1_col[cname] = (ct <= 1.0).mean() * 100 if len(ct) >= SUPPRESSION_THRESHOLD else None
        median_by_col = {1: cohort_medians[f'Historical (<{split_yr})'],
                          2: cohort_medians[f'Recent (≥{split_yr})']}
        yr1_by_col = {1: pct_by_yr1_col[f'Historical (<{split_yr})'],
                      2: pct_by_yr1_col[f'Recent (≥{split_yr})']}
    else:
        median_by_col = {1: median_cancel_y}
        yr1_by_col = {1: (cancelled_times <= 1.0).mean() * 100
                      if len(cancelled_times) >= SUPPRESSION_THRESHOLD else None}

    hist_med = median_by_col.get(1)
    rec_med  = median_by_col.get(2) if show_cohort_split else None

    # ── Comparison + savings text — attached to the RECENT panel's box only ─
    timing_text_by_col  = {1: '', 2: ''}
    savings_text_by_col = {1: '', 2: ''}
    if show_cohort_split and hist_med is not None and rec_med is not None:
        delta        = rec_med - hist_med
        delta_months = round(abs(delta) * 12)
        arrow        = '▲' if delta >= 0 else '▼'
        direction    = '<b>later</b>' if delta >= 0 else '<b>earlier</b>'

        hist_cost = estimate_sunk_cost(hist_med)
        rec_cost  = estimate_sunk_cost(rec_med)
        cost_diff = hist_cost - rec_cost   # positive = recent cohort costs less
        cost_word = 'less' if cost_diff >= 0 else 'more'

        timing_text_by_col[2] = (
            f'<br><span style="font-size:10px">{arrow} {delta_months} months {direction} '
            f'than historical median</span>'
        )
        savings_text_by_col[2] = (
            f'<br><span style="font-size:10px">≈${abs(cost_diff):,.0f}/apprentice <b>{cost_word}</b> '
            f'in training wages</span>'
        )

    for col in range(1, n_cols + 1):
        if pd.notna(expected_years):
            fig.add_vline(
                x=float(expected_years), line_dash='dot',
                line_color='#E2E8F0', line_width=1,
                col=col,  # type: ignore[arg-type]
            )

        col_median = median_by_col.get(col)
        if col_median is not None and col_median <= x_max:
            sunk_cost      = estimate_sunk_cost(col_median)
            sunk_cost_text = f'<br><span style="font-size:10px">≈${sunk_cost:,.0f}/apprentice in training wages</span>'

            pct_yr1 = yr1_by_col.get(col)
            year1_text = ''
            if pct_yr1 is not None:
                year1_text = (
                    f'<br><span style="font-size:10px">{pct_yr1:.0f}% of cancellations '
                    f'had happened by year 1</span>'
                )

            fig.add_vline(
                x=col_median, line_dash='dash', line_color='#B91C1C',
                line_width=1.2, opacity=0.7, col=col,  # type: ignore[arg-type]
                annotation_text=(
                    f'Half (50%) had cancelled by year {col_median:.1f}'
                    f'{year1_text}'
                    f'{timing_text_by_col.get(col, "")}'
                    f'{sunk_cost_text}'
                    f'{savings_text_by_col.get(col, "")}'
                ),
                annotation=dict(
                    bgcolor='rgba(255,255,255,0.9)',
                    bordercolor='#B91C1C',
                    borderpad=4,
                    font=dict(size=11, color='#B91C1C'),
                    y=0.73,
                    yref="paper",
                    xanchor='center'     
                )
            )
            
    # ── Layout ──────────────────────────────────────────────────────────────
    fig.update_xaxes(title_text='Years since enrollment', range=[0, x_max])
    fig.update_yaxes(title_text='Cumulative share (%)', ticksuffix='%', range=[0, 100])
    if show_cohort_split:
        fig.update_yaxes(title_text='', row=1, col=2)

    fig.add_annotation(
        text='Hover over lines for year-by-year detail',
        xref='paper', yref='paper',
        x=0.99, y=-0.30,
        xanchor='right', yanchor='top',
        showarrow=False,
        font=dict(size=11, color='#9CA3AF', family='Inter, system-ui, Arial, sans-serif'),
    )

    fig.add_annotation(
        text=(
            '<b style="color:#1E40AF">EXECUTIVE RECOMMENDATION:</b> '
            '<span style="color:#374151">'
            '<br><b>BLUF:</b> Early exits are a diagnostic signal, <b>not a KPI</b>. '
            'The real opportunity is reducing preventable late exits, which are more costly and represent avoidable loss of talent.<br><br>'
            '<b>Focus on:</b><br>'
            '• Identify the drivers of cancellations between Years 1–3, where most attrition occurs.<br>'
            '• Audit supervision and jobsite placement quality during the transition to higher responsibility.<br>'
            '• Benchmark against programs with stronger mid‑program retention to identify practices worth adopting.<br><br>'

            '<b>Do not optimize for earlier exits.</b> The goal is better matching and support, not accelerating attrition.</b><br>'

            'Note: Annual completion rates reflect outcomes from multiple older cohorts. Survival curves isolate new cohorts only, which have not yet reached the completion window.'
            '</span>'
        ),

        xref='paper', yref='paper',
        x=0.5, y=-0.42,                  # Changed x from 0.0 to 0.5
        xanchor='center', yanchor='top', # Changed xanchor from 'left' to 'center'
        showarrow=False, 
        align='left',                  # Optional: Changed text alignment inside the box from 'left' to 'center'
        font=dict(size=15, family='Inter, system-ui, Arial, sans-serif'),
        bgcolor='rgba(30,64,175,0.06)',
        bordercolor='#1E40AF', borderwidth=1, borderpad=8,

    )


    fig.update_layout(
        title=(
            f'<b>When do apprentices leave or finish?</b><br>'
            f'<span style="font-size:12px;color:#4A4D59">'
            f'{label} · n={total_n:,} · Expected completion: {expected_years:.1f} years · '
            f'Wage assumed at ${ASSUMED_APPRENTICE_WAGE}/hr, {ASSUMED_HOURS_PER_YEAR} hrs/yr — illustrative only</span>'
            #f'${ASSUMED_APPRENTICE_WAGE}/hr × {ASSUMED_HOURS_PER_YEAR} hrs/yr (assumed, not actual payroll)</span>'
        ),
        template='simple_white',
        font=dict(family='Inter, system-ui, Arial, sans-serif', size=13, color='#374151'),
        legend=dict(orientation='h', x=0.5, xanchor='center', y=-0.25),
        autosize=True, height=750, margin=dict(t=130, b=350),
        #autosize=True, height=460, margin=dict(t=120, b=130),
    )
    return fig

In [33]:
def _build_demographics_fig(df, occ):
    total_n = len(df)
    if total_n < SUPPRESSION_THRESHOLD:
        return _empty_state_fig(f'<b>{occ}</b>')

    # -----------------------------
    # Suppression logic
    # -----------------------------
    age_counts = df['AGE_COHORT'].value_counts()
    safe_age = age_counts[age_counts >= SUPPRESSION_THRESHOLD].copy()
    suppressed_age = age_counts[age_counts < SUPPRESSION_THRESHOLD].sum()
    if suppressed_age > 0:
        safe_age['Other / Below reporting threshold'] = suppressed_age


    race_counts = df['RACE_AND_ETHNICITY'].value_counts()
    safe_race = race_counts[race_counts >= SUPPRESSION_THRESHOLD].copy()
    suppressed_race = race_counts[race_counts < SUPPRESSION_THRESHOLD].sum()
    if suppressed_race > 0:
        safe_race['Other / Below reporting threshold'] = suppressed_race
    safe_race = safe_race.sort_values(ascending=True)

    # -----------------------------
    # Color palettes
    # -----------------------------
    # Donut (age_cohort)
    donut_colors = ['#F4A261', '#E76F51', '#D3D3D3']
    donut_colors = ['#FDBA74', '#F9A8D4', '#E5E7EB']

    # Race palette
    race_palette = {
        'Native Hawaiian or Pacific Islander': '#0891B2',
        'Asian': '#1D4ED8',
        'White': '#3B82F6',
        'Hispanic or Latino': '#0EA5E9',
        'Multiracial': '#38BDF8',
        'Black or African American': '#1E3A8A',
        'American Indian or Alaska Native': '#0D9488',
        'Participant Did Not Self-Identify': '#CBD5E1',
        'Other / Below reporting threshold': '#94A3B8'
    }
    bar_colors = [race_palette.get(c, '#CBD5E1') for c in safe_race.index]

    # -----------------------------
    # Subplots
    # -----------------------------
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{'type': 'xy'}, {'type': 'domain'}]], 
        column_widths=[0.68, 0.32],
        subplot_titles=['By Race & Ethnicity', 'By Age Cohort'],
        horizontal_spacing=0.08,
    )

    fig.update_annotations(font=dict(size=14))

    # -----------------------------
    # Pie Chart 
    # -----------------------------
    fig.add_trace(go.Pie(
        labels=safe_age.index,
        values=safe_age.values,
        sort=False,
        # domain=dict(x=[0.70, 1.00], y=[0.20, 0.80]),
        # hole=0.50,
        domain=dict(x=[0.72, 1.00], y=[0.25, 0.75]),
        hole=0.55,
        marker=dict(colors=donut_colors, line=dict(color='white', width=2)),
        textinfo='percent+label',
        textposition='inside',
        hovertemplate='<b>%{label}</b><br>%{value:,} apprentices (%{percent})<extra></extra>',
    ), row=1, col=2)

    # -----------------------------
    # 5. Bar Chart 
    # -----------------------------
    fig.add_trace(go.Bar(
        x=safe_race.values,
        y=safe_race.index,
        orientation='h',
        marker_color=bar_colors,
        text=[f'{v:,} ({v/total_n*100:.0f}%)' for v in safe_race.values],
        textposition='outside',
        hovertemplate='<b>%{y}</b><br>%{x:,} apprentices<extra></extra>',
    ), row=1, col=1)

    fig.update_xaxes(
        title_text='Apprentices',
        showgrid=True,
        gridcolor='#E5E7EB',
        range=[0, safe_race.values.max() * 1.18],
        row=1, col=1
    )
    fig.update_yaxes(tickfont=dict(size=12))

    # -----------------------------
    # 6. Layout
    # -----------------------------
    fig.update_layout(
        title=(
            f'<b>Demographics</b><br>'
            f'<span style="font-size:13px;color:#6B7280">'
            f'{occ} · n={total_n:,} · categories below {SUPPRESSION_THRESHOLD} grouped'
            f'</span>'
        ),
        template='simple_white',
        font=dict(
            family='Inter, system-ui, Arial, sans-serif',
            size=13,
            color='#374151'
        ),
        plot_bgcolor='rgba(250,250,250,1)',
        paper_bgcolor='white',
        margin=dict(t=110, b=40, l=40, r=40),
        autosize=True,
        height=460,
        showlegend=False,
        uniformtext_minsize=10,
        uniformtext_mode='hide'
    )

    return fig


In [34]:
# ── Clickable, occupation-filterable funnel widget — same standardized chart
# as Section 8, but as a FigureWidget wired to the dropdowns below in BOTH
# directions:
#   - Click a point  -> drills into that program (sets occ + program dropdowns)
#   - Change Occupation dropdown -> the funnel itself re-filters to that occupation
#
# The click handler reads PROGRAM_NAME/OCCUPATION_GROUP_NAME straight off the
# trace's own customdata at click time (not from a captured DataFrame), so it
# never goes stale after update_funnel_widget() re-filters the trace data.
#
# Safe by construction: every program shown here cleared MIN_N (≥20), and
# MIN_N > SUPPRESSION_THRESHOLD (≥10), so every clickable point is guaranteed
# to exist as its own named option in the dropdowns below, never folded into
# an "(combined)" bucket.

def _funnel_subset(occ):
    if occ in ('All Trades', 'Other Occupations (combined)'):
        return hi_occ_metrics
    return hi_occ_metrics[hi_occ_metrics['OCCUPATION_GROUP_NAME'] == occ]

def _funnel_click_handler(trace, points, state):
    if not points.point_inds:
        return
    idx = points.point_inds[0]
    prog_name, occ_name = trace.customdata[idx][0], trace.customdata[idx][1]
    try:
        occ_dropdown.value     = occ_name
        program_dropdown.value = prog_name
    except Exception:
        pass  # defensive — see safety note above

def build_funnel_widget():
    n_max = hi_occ_metrics['total_apprentices'].max()
    fig = go.FigureWidget()

    fig.add_trace(go.Scatter(
        x=[MIN_N, n_max, n_max, MIN_N], y=[3, 3, -3, -3],
        fill='toself', fillcolor='rgba(255,200,200,0.25)',
        line=dict(color='rgba(0,0,0,0)'), name='±3 SD (~99.7%)', hoverinfo='skip',
    ))
    fig.add_trace(go.Scatter(
        x=[MIN_N, n_max, n_max, MIN_N], y=[2, 2, -2, -2],
        fill='toself', fillcolor='rgba(200,230,200,0.35)',
        line=dict(color='rgba(0,0,0,0)'), name='±2 SD (~95.4%)', hoverinfo='skip',
    ))

    for ptype, color in TYPE_COLORS.items():
        sub = hi_occ_metrics[hi_occ_metrics['program_type'] == ptype].reset_index(drop=True)
        if sub.empty:
            continue
        fig.add_trace(go.Scatter(
            x=sub['total_apprentices'], y=sub['z_score'],
            mode='markers', name=ptype,
            marker=dict(size=9, color=color, opacity=0.8),
            customdata=sub[['PROGRAM_NAME', 'OCCUPATION_GROUP_NAME',
                            'completion_rate', 'national_avg_completion_rate']].values,
            hovertemplate=(
                '<b>%{customdata[0]}</b><br>'
                'Occupation: %{customdata[1]}<br>'
                'Size: %{x}<br>'
                'Completion: %{customdata[2]:.1f}% (national: %{customdata[3]:.1f}%)<br>'
                'Standardized deviation: %{y:.1f} SD<br>'
                '<i>Click to view full pipeline breakdown</i>'
                '<extra></extra>'
            ),
        ))
        fig.data[-1].on_click(_funnel_click_handler)

    fig.add_hline(y=0, line_dash='dash', line_color='grey',
                  annotation_text="Matches national average for that program's trade")
    fig.update_layout(
        title=(
            f'{scope_label} Funnel Plot: Standardized Completion Rate vs Program Size<br>'
            "<sup>All occupations · Click any point to view that program's pipeline breakdown below.</sup>"
        ),
        xaxis_title='Total apprentices',
        yaxis_title='Standardized deviation from national benchmark (SD)',
        template='simple_white', autosize=True,  width=1000, height=500,
    )
    return fig

funnel_widget = build_funnel_widget()

def update_funnel_widget(occ):
    sub_all = _funnel_subset(occ)
    n_max   = max(sub_all['total_apprentices'].max(), MIN_N + 10) if len(sub_all) else MIN_N + 10
    scope_text = 'All occupations' if occ in ('All Trades', 'Other Occupations (combined)') else occ

    with funnel_widget.batch_update():
        for trace in funnel_widget.data:
            if trace.name in ('±3 SD (~99.7%)', '±2 SD (~95.4%)'):
                trace.x = [MIN_N, n_max, n_max, MIN_N]
            elif trace.name in TYPE_COLORS:
                sub = sub_all[sub_all['program_type'] == trace.name]
                trace.x = sub['total_apprentices']
                trace.y = sub['z_score']
                trace.customdata = sub[['PROGRAM_NAME', 'OCCUPATION_GROUP_NAME',
                                         'completion_rate', 'national_avg_completion_rate']].values
        funnel_widget.layout.title.text = (
            f'Funnel Plot: Standardized Completion Rate vs Program Size — {scope_label}<br>'
            f"<sup>{scope_text} · Click any point to view that program's pipeline breakdown below.</sup>"
        )

# ── Widgets ──────────────────────────────────────────────────────────────
occ_dropdown = widgets.Dropdown(
    options=get_occupation_options(), value='All Trades',
    description='Occupation:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='420px'),
)
program_dropdown = widgets.Dropdown(
    options=get_programs_for_occ('All Trades'), value='All Programs',
    description='Program:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='420px'),
)

sankey_out   = widgets.Output()
survival_out = widgets.Output()
trend_out    = widgets.Output()
enrol_out    = widgets.Output()
demo_out     = widgets.Output()   

def refresh():
    occ     = occ_dropdown.value
    program = program_dropdown.value
    df      = get_filtered(occ, program)
    label   = occ if program == 'All Programs' else f'{occ} · {program}'

    if is_suppressed(df):
        fig = _empty_state_fig(f'<b>{label}</b>')
        with sankey_out:   clear_output(wait=True); fig.show(config={'responsive': True})
        with survival_out: clear_output(wait=True); fig.show(config={'responsive': True})
        with trend_out:    clear_output(wait=True); fig.show(config={'responsive': True})
        with enrol_out:    clear_output(wait=True); fig.show(config={'responsive': True})
        with demo_out: clear_output(wait=True); fig.show(config={'responsive': True})
        return

    with sankey_out:
        clear_output(wait=True)
        _build_sankey_fig(df, label).show(config={'responsive': True})
    with survival_out:
        clear_output(wait=True)
        build_program_survival_fig(df, label).show(config={'responsive': True})
    with trend_out:
        clear_output(wait=True)
        _build_trend_fig(df, label).show(config={'responsive': True})
    with enrol_out:
        clear_output(wait=True)
        _build_enrollment_fig(df, label).show(config={'responsive': True})
    with demo_out:
        clear_output(wait=True)
        _build_demographics_fig(df, label).show(config={'responsive': True})

def on_occ_change(change):
    if change['name'] != 'value':
        return
    new_occ = change['new']
    new_programs = get_programs_for_occ(new_occ)
    program_dropdown.unobserve(on_program_change, names='value')
    program_dropdown.options = new_programs
    program_dropdown.value   = 'All Programs'
    program_dropdown.observe(on_program_change, names='value')
    update_funnel_widget(new_occ)
    refresh()

def on_program_change(change):
    if change['name'] != 'value':
        return
    refresh()

occ_dropdown.observe(on_occ_change, names='value')
program_dropdown.observe(on_program_change, names='value')

# ── Initial render ─────────────────────────────────────────────────────────
refresh()

display(widgets.VBox(
    [funnel_widget,
     widgets.HTML('<p style="font-family:Inter,system-ui,sans-serif;font-size:13px;'
                  'color:#6B7280;margin:16px 0 4px">Or filter manually by occupation and program</p>'),
     widgets.HBox([occ_dropdown, program_dropdown], layout=widgets.Layout(gap='16px')),
     sankey_out, survival_out, trend_out, enrol_out, demo_out],
    layout=widgets.Layout(gap='20px'),
))


    'data': [{'fill': 'toself',
              'fillcolor': 'rgba(255,200,200,0.2…

## 11. Data Reliability Diagnostic

The chart below tests if programs are under-reporting their active apprentices numbers


In [35]:
def build_active_reporting_fig(df, valid_programs):
    g = df.groupby(['OCCUPATION_GROUP_NAME', 'PROGRAM_NAME'])
    diag = g.agg(
        total=('APPRENTICE_STATUS_CODE', 'size'),
        n_active=('APPRENTICE_STATUS_CODE', lambda x: (x == 'Active').sum()),
        n_suspended=('APPRENTICE_STATUS_CODE', lambda x: (x == 'Suspended').sum()),
        max_enroll_year=('FISCAL_YEAR', 'max'),
    ).reset_index() 
    
    # Restrict to exactly the programs already used everywhere else in the
    # dashboard (hi_occ_metrics, filtered to total_apprentices >= MIN_N) — via
    # a join on valid_programs rather than re-deriving the filter by hand, so
    # this chart's population can never silently drift out of sync with the
    # rest of the deck.
    diag = diag.merge(
        valid_programs[['OCCUPATION_GROUP_NAME', 'PROGRAM_NAME']],
        on=['OCCUPATION_GROUP_NAME', 'PROGRAM_NAME'], how='inner',
    )
    diag = diag[diag['total'] >= MIN_N]  
    diag['unresolved'] = diag['n_active'] + diag['n_suspended']

    recent_cutoff = df['FISCAL_YEAR'].max() - 1
    conditions = [
        diag['unresolved'] > 0,
        (diag['unresolved'] == 0) & (diag['max_enroll_year'] >= recent_cutoff),
    ]
    choices = ['Reports active apprentices', 'Zero active — despite recent enrollment']
    diag['report_status'] = np.select(conditions, choices, default='Zero active — no recent enrollment')

    counts = diag['report_status'].value_counts()
    pct = (counts / counts.sum() * 100).round(1)

    fig = go.Figure(go.Bar(
        y=counts.values, x=counts.index, orientation='v',
        marker_color=['#15803D', '#B91C1C', '#94A3B8'],
        text=[f'{c} programs ({p}%)' for c, p in zip(counts.values, pct.values)],
        textposition='outside',
    ))
    fig.update_layout(
        title=(f'<b>Do programs report current enrollment?</b><br>'
               f'<span style="font-size:12px;color:#6B7280">'
               #f'Of {len(diag)} programs with ≥{MIN_N} apprentices · '
               f'{pct.get("Zero active — despite recent enrollment", 0):.0f}% report zero '
               f'active apprentices despite enrolling someone in the last year</span>'),
        xaxis_title='Number of programs',
        template='simple_white', autosize=True, height=450,
        margin=dict(l=100, t=90, b=40, r=80),
    )
    fig.show()
    return diag

reporting_diag = build_active_reporting_fig(active_data, hi_occ_metrics)

## 12. Correlation

Meant to view the correlation between cancellation rate and completion rate

In [36]:
def _program_occ_completion(df):
    """Program x occupation completion rate, computed fresh and filtered to MIN_N —
    self-contained so the same function works on either Hawaiʻi or national data."""
    g = df.groupby(['OCCUPATION_GROUP_NAME', 'PROGRAM_NAME']).agg(
        total_apprentices=('APPRENTICE_STATUS_CODE', 'size'),
        completed=('APPRENTICE_STATUS_CODE', lambda x: (x == 'Completed').sum()),
        cancelled=('APPRENTICE_STATUS_CODE', lambda x: (x == 'Cancelled').sum()),
        min_year=('FISCAL_YEAR', 'min'),
        max_year=('FISCAL_YEAR', 'max'),
    ).reset_index()
    g = g[g['total_apprentices'] >= MIN_N].copy()
    resolved = g['completed'] + g['cancelled']
    g['completion_rate'] = np.where(resolved > 0, g['completed'] / resolved * 100, np.nan)
    g['years_enrolling'] = g['max_year'] - g['min_year'] + 1
    return g.dropna(subset=['completion_rate', 'years_enrolling'])

def _fit_ols(x, y):
    slope, intercept = np.polyfit(x, y, 1)
    y_pred = slope * x + intercept
    ss_res, ss_tot = np.sum((y - y_pred) ** 2), np.sum((y - y.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return slope, intercept, r2

def build_tenure_vs_completion_fig(hi_df, nat_df):
    hi_g, nat_g = _program_occ_completion(hi_df), _program_occ_completion(nat_df)

    fig = go.Figure()
    summary = {}
    for label, g, color in [('Hawaiʻi', hi_g, '#1E40AF'), ('National', nat_g, '#94A3B8')]:
        x, y = g['years_enrolling'].values.astype(float), g['completion_rate'].values.astype(float)
        slope, intercept, r2 = _fit_ols(x, y)
        summary[label] = (slope, r2, len(g))

        fig.add_trace(go.Scatter(
            x=x, y=y, mode='markers', name=f'{label} programs',
            marker=dict(size=7, color=color, opacity=0.45 if label == 'National' else 0.6),
            text=g['PROGRAM_NAME'] + ' · ' + g['OCCUPATION_GROUP_NAME'],
            hovertemplate='<b>%{text}</b><br>Years enrolling: %{x}<br>Completion rate: %{y:.1f}%<extra></extra>',
        ))
        x_line = np.linspace(x.min(), x.max(), 100)
        fig.add_trace(go.Scatter(
            x=x_line, y=slope * x_line + intercept, mode='lines',
            name=f'{label} OLS fit (R²={r2:.3f})',
            line=dict(color=color, width=2.5, dash='dash'),
        ))

    fig.update_layout(
        title=(
            f'<b>Program Tenure vs. Completion Rate — Hawaiʻi vs. National</b><br>'
            f'<span style="font-size:12px;color:#6B7280">'
            f'Hawaiʻi: n={summary["Hawaiʻi"][2]}, slope={summary["Hawaiʻi"][0]:.2f} pp/yr, R²={summary["Hawaiʻi"][1]:.3f}  ·  '
            f'National: n={summary["National"][2]}, slope={summary["National"][0]:.2f} pp/yr, R²={summary["National"][1]:.3f}<br>'
            f'Correlational only — younger programs carry more unresolved apprentices, which can '
            f'inflate completion_rate independent of program quality. A similar pattern in both '
            f'lines points to this effect rather than something Hawaiʻi-specific.</span>'
        ),
        xaxis_title='Years enrolling apprentices (program tenure)',
        yaxis_title='Completion rate (%)',
        template='simple_white', autosize=True, height=520,
        legend=dict(orientation='h', x=0.5, xanchor='center', y=-0.18),
        margin=dict(t=120, b=100),
    )
    fig.show()
    return hi_g, nat_g, summary

hi_tenure, nat_tenure, tenure_summary = build_tenure_vs_completion_fig(active_data, active_nat)

In [37]:
def compute_early_cancellation_metrics(df):
    d = df[df['APPRENTICE_STATUS_CODE'].isin(['Completed', 'Cancelled'])].copy()
    d['years_since_enrollment'] = d['duration_days'] / 365.25   # same basis as survival analysis

    g = d.groupby(['OCCUPATION_GROUP_NAME', 'PROGRAM_NAME']).agg(
        total=('APPRENTICE_STATUS_CODE', 'size'),
        completed=('APPRENTICE_STATUS_CODE', lambda x: (x == 'Completed').sum()),
        cancelled=('APPRENTICE_STATUS_CODE', lambda x: (x == 'Cancelled').sum()),
        median_cancel_time=('years_since_enrollment',
            lambda x: x[d.loc[x.index, 'APPRENTICE_STATUS_CODE'] == 'Cancelled'].median()),
        share_cancel_year1=('years_since_enrollment',
            lambda x: ((d.loc[x.index, 'APPRENTICE_STATUS_CODE'].eq('Cancelled')) & (x <= 1)).sum()
                      / (d.loc[x.index, 'APPRENTICE_STATUS_CODE'] == 'Cancelled').sum()
                      if (d.loc[x.index, 'APPRENTICE_STATUS_CODE'] == 'Cancelled').sum() > 0 else np.nan),
        survived_year1=('years_since_enrollment', lambda x: (x > 1).sum()),
        completed_after_year1=('APPRENTICE_STATUS_CODE',
            lambda x: ((x == 'Completed') & (d.loc[x.index, 'years_since_enrollment'] > 1)).sum()),
    ).reset_index()

    g = g[g['total'] >= MIN_N].copy()   # consistency with every other chart in the deck
    resolved_n = g['completed'] + g['cancelled']
    g['completion_rate'] = np.where(resolved_n > 0, g['completed'] / resolved_n * 100, np.nan)
    g['completion_rate_after_year1'] = (
        g['completed_after_year1'] / g['survived_year1'].replace(0, np.nan)
    ) * 100
    return g

g = compute_early_cancellation_metrics(active_data)

x = g['median_cancel_time'].values
y = g['completion_rate_after_year1'].values
valid = ~np.isnan(x) & ~np.isnan(y)
x, y = x[valid], y[valid]

slope, intercept = np.polyfit(x, y, 1)
r2 = 1 - np.sum((y - (slope*x+intercept))**2) / np.sum((y - y.mean())**2)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=y, mode='markers', marker=dict(size=8, color='#1E40AF', opacity=0.6)))
x_line = np.linspace(x.min(), x.max(), 100)
fig.add_trace(go.Scatter(x=x_line, y=slope*x_line+intercept, mode='lines',
                          line=dict(color='#B91C1C', dash='dash'), name=f'OLS fit (R²={r2:.3f})'))
fig.update_layout(
    title=f'Median Cancellation Timing vs. Completion Rate After Year 1 (R²={r2:.3f})',
    xaxis_title='Median cancellation time (years)',
    yaxis_title='Completion rate, conditional on surviving year 1 (%)',
    template='simple_white',
)
fig.show()

In [38]:
resolved = active_data[active_data['APPRENTICE_STATUS_CODE'].isin(['Completed', 'Cancelled'])].copy()
resolved['start_year'] = pd.to_datetime(resolved['START_DATE']).dt.year
resolved['exit_year']  = pd.to_datetime(resolved['EXIT_DATE']).dt.year

print('FISCAL_YEAR matches START year:', (resolved['FISCAL_YEAR'] == resolved['start_year']).mean())
print('FISCAL_YEAR matches EXIT year:', (resolved['FISCAL_YEAR'] == resolved['exit_year']).mean())

FISCAL_YEAR matches START year: 0.0014349775784753362
FISCAL_YEAR matches EXIT year: 0.22617124394184168


## 13. Export

In [35]:
hawaii_merged.to_parquet('../data/processed/hawaii_merged.parquet', index=False)
hi_trades.to_parquet('../data/processed/hawaii_trades.parquet', index=False)
hi_metrics_by_program.to_parquet('../data/processed/hawaii_program_metrics.parquet', index=False)
hi_occ_metrics.to_parquet('../data/processed/hawaii_occ_metrics.parquet', index=False)

print("Exported:")
print("  hawaii_merged.parquet")
print("  hawaii_trades.parquet")
print("  hawaii_program_metrics.parquet")
print("  hawaii_occ_metrics.parquet")


Exported:
  hawaii_merged.parquet
  hawaii_trades.parquet
  hawaii_program_metrics.parquet
  hawaii_occ_metrics.parquet
